In [1]:
import jax
print(jax.devices())  # Should show CUDA devices if GPU available

[CpuDevice(id=0)]


In [4]:
# Cell 1: Verify .X is raw counts
import anndata as ad
snrna = ad.read_h5ad("/Users/sydneycole/neuro/neuro/data/raw/snrna.h5ad")  # adjust path

print("Shape:", snrna.shape)
print("\nFirst 10x10 of .X:")
print(snrna.X[:10, :10].toarray())
print("\nMax value in first 1000 cells:", snrna.X[:1000].max())
print("Are values integers?", (snrna.X[:1000].data % 1 == 0).all())
print("\nLayers available:", list(snrna.layers.keys()))
print(".raw is None:", snrna.raw is None)

Shape: (65737, 30968)

First 10x10 of .X:
[[0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1]]

Max value in first 1000 cells: 945
Are values integers? True

Layers available: []
.raw is None: True


In [5]:
# Cell 2: Verify MERFISH gene overlap
merfish = ad.read_h5ad("/Users/sydneycole/neuro/neuro/data/raw/merscope_integrated_855.h5ad")  # adjust

merfish_genes = set(merfish.var_names)
snrna_genes = set(snrna.var_names)
overlap = merfish_genes & snrna_genes
missing = merfish_genes - snrna_genes

print(f"MERFISH panel size: {len(merfish_genes)}")
print(f"snRNA gene count: {len(snrna_genes)}")
print(f"Overlap: {len(overlap)} / {len(merfish_genes)} ({100*len(overlap)/len(merfish_genes):.1f}%)")
print(f"\nMissing from snRNA ({len(missing)}):")
print(sorted(missing)[:30])

MERFISH panel size: 300
snRNA gene count: 30968
Overlap: 296 / 300 (98.7%)

Missing from snRNA (4):
['DENND2B', 'ELAPOR1', 'FLJ20021', 'PMCHL2']


/opt/anaconda3/envs/neurospatial/lib/python3.11/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [7]:
for alias in ['ST5', 'KIAA1324', 'TMEM131L', 'PMCHL2']:
    print(f"{alias}: {'FOUND' if alias in snrna.var_names else 'not found'}")

# Reconcile HGNC alias drift between MERFISH panel and snRNA reference
# DENND2B, ELAPOR1, FLJ20021 are older symbols; snRNA uses current HGNC names
# PMCHL2 is a pseudogene not present in snRNA reference — will be dropped by ENVI's overlap step

alias_map = {
    'DENND2B':  'ST5',
    'ELAPOR1':  'KIAA1324',
    'FLJ20021': 'TMEM131L',
}

# Sanity check: confirm old names exist in MERFISH and new names don't (no collisions)
for old, new in alias_map.items():
    assert old in merfish.var_names, f"{old} not in MERFISH panel"
    assert new not in merfish.var_names, f"{new} already in MERFISH panel — would collide"

merfish.var_names = [alias_map.get(g, g) for g in merfish.var_names]

# Verify the rename worked
new_overlap = set(merfish.var_names) & set(snrna.var_names)
print(f"Overlap after rename: {len(new_overlap)} / {len(merfish.var_names)}")
# Expect: 299 / 300

ST5: FOUND
KIAA1324: FOUND
TMEM131L: FOUND
PMCHL2: not found
Overlap after rename: 299 / 300
